# An?lisis de Supervivencia del Titanic con Regresi?n Log?stica

Este cuaderno desarrolla un flujo completo de anal?tica de datos sobre el archivo `tested.csv`, que contiene informaci?n de pasajeros del Titanic y una variable objetivo llamada `Survived`.

El objetivo es construir un an?lisis ordenado, reproducible y f?cil de explicar: primero entendemos los datos, luego hacemos limpieza y exploraci?n, despu?s preparamos variables para modelado y finalmente entrenamos un modelo de clasificaci?n con regresi?n log?stica.


## 1. Contexto del conjunto de datos

Cada fila representa un pasajero. Las columnas combinan informaci?n demogr?fica, datos del viaje y la etiqueta de supervivencia.

| Columna | Descripci?n | Tipo esperado |
|---|---|---|
| `PassengerId` | Identificador ?nico del pasajero | Num?rico |
| `Survived` | Variable objetivo: 0 = no sobrevivi?, 1 = sobrevivi? | Categ?rica/binaria |
| `Pclass` | Clase del boleto: 1 = alta, 2 = media, 3 = baja | Categ?rica ordinal |
| `Name` | Nombre completo del pasajero | Texto |
| `Sex` | Sexo registrado del pasajero | Categ?rica |
| `Age` | Edad en a?os | Num?rica |
| `SibSp` | N?mero de hermanos/c?nyuges a bordo | Num?rica discreta |
| `Parch` | N?mero de padres/hijos a bordo | Num?rica discreta |
| `Ticket` | C?digo del boleto | Texto |
| `Fare` | Tarifa pagada | Num?rica |
| `Cabin` | Cabina asignada | Texto/categ?rica |
| `Embarked` | Puerto de embarque: C = Cherbourg, Q = Queenstown, S = Southampton | Categ?rica |

> Nota anal?tica: este archivo contiene 418 registros. En algunas versiones p?blicas del dataset del Titanic, el archivo de prueba no trae la supervivencia real; aqu? s? aparece una columna `Survived`. Por eso validaremos su comportamiento antes de interpretar el modelo como si fuera una medici?n hist?rica definitiva.


In [ ]:
# 2. Importar librer?as
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
    RocCurveDisplay,
)
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

sns.set_theme(style="whitegrid", palette="Set2")
pd.set_option("display.max_columns", 50)


In [ ]:
# 3. Cargar datos
DATA_PATH = Path("tested.csv")

if not DATA_PATH.exists():
    raise FileNotFoundError(f"No se encontr? el archivo {DATA_PATH.resolve()}")

df = pd.read_csv(DATA_PATH)
print(f"Filas: {df.shape[0]:,} | Columnas: {df.shape[1]:,}")
df.head(10)


## 4. Primera lectura de la tabla

Antes de modelar, conviene revisar tres aspectos b?sicos:

- El tama?o del dataset.
- El tipo de dato de cada columna.
- La presencia de valores nulos, porque estos pueden afectar el an?lisis y el entrenamiento del modelo.


In [ ]:
# Estructura general
df.info()


In [ ]:
# Resumen de valores faltantes
missing_summary = (
    df.isna()
    .sum()
    .to_frame("valores_faltantes")
    .assign(porcentaje=lambda x: (x["valores_faltantes"] / len(df) * 100).round(2))
    .sort_values("valores_faltantes", ascending=False)
)
missing_summary


In [ ]:
# Estad?sticos descriptivos para variables num?ricas
df.describe().T


## 5. Calidad y lectura cr?tica de la variable objetivo

La variable `Survived` es la etiqueta que queremos predecir. Antes de entrenar, revisamos su distribuci?n y su relaci?n con algunas variables clave. Esto ayuda a detectar clases desbalanceadas, posibles sesgos o patrones demasiado perfectos para ser reales.


In [ ]:
# Distribuci?n de la variable objetivo
survival_counts = df["Survived"].value_counts().rename(index={0: "No sobrevivi?", 1: "Sobrevivi?"})
survival_rate = df["Survived"].mean()

print(survival_counts)
print(f"\nTasa de supervivencia observada: {survival_rate:.2%}")


In [ ]:
# Relaci?n entre sexo registrado y supervivencia
pd.crosstab(df["Sex"], df["Survived"], margins=True, normalize="index").round(3)


**Observaci?n importante:** si una variable explica casi perfectamente la etiqueta, el modelo puede obtener m?tricas muy altas sin haber aprendido una relaci?n generalizable. En anal?tica de datos esto se revisa como posible fuga de informaci?n, etiqueta sint?tica o estructura propia del archivo.


## 6. An?lisis exploratorio de datos

Ahora observamos patrones de supervivencia por clase, sexo, edad, tarifa y puerto de embarque. Estas visualizaciones ayudan a convertir una tabla en conocimiento accionable.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

sns.countplot(data=df, x="Survived", ax=axes[0, 0])
axes[0, 0].set_title("Distribuci?n de supervivencia")
axes[0, 0].set_xticklabels(["No", "S?"])
axes[0, 0].set_xlabel("Sobrevivi?")
axes[0, 0].set_ylabel("Cantidad")

sns.barplot(data=df, x="Pclass", y="Survived", ax=axes[0, 1])
axes[0, 1].set_title("Tasa de supervivencia por clase")
axes[0, 1].set_xlabel("Clase")
axes[0, 1].set_ylabel("Tasa de supervivencia")

sns.barplot(data=df, x="Sex", y="Survived", ax=axes[1, 0])
axes[1, 0].set_title("Tasa de supervivencia por sexo registrado")
axes[1, 0].set_xlabel("Sexo")
axes[1, 0].set_ylabel("Tasa de supervivencia")

sns.barplot(data=df, x="Embarked", y="Survived", ax=axes[1, 1])
axes[1, 1].set_title("Tasa de supervivencia por puerto de embarque")
axes[1, 1].set_xlabel("Puerto")
axes[1, 1].set_ylabel("Tasa de supervivencia")

plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(data=df, x="Age", hue="Survived", kde=True, bins=25, ax=axes[0])
axes[0].set_title("Distribuci?n de edad seg?n supervivencia")
axes[0].set_xlabel("Edad")

sns.boxplot(data=df, x="Survived", y="Fare", ax=axes[1])
axes[1].set_title("Tarifa pagada seg?n supervivencia")
axes[1].set_xlabel("Sobrevivi?")
axes[1].set_ylabel("Tarifa")
axes[1].set_xticklabels(["No", "S?"])

plt.tight_layout()
plt.show()


## 7. Ingenier?a de variables

Adem?s de usar las columnas originales, creamos variables derivadas que suelen aportar se?al en este problema:

- `Title`: t?tulo extra?do del nombre, como Mr, Mrs, Miss, Master.
- `FamilySize`: tama?o del grupo familiar a bordo.
- `IsAlone`: indicador de pasajero sin familiares registrados.
- `FarePerPerson`: tarifa aproximada por integrante del grupo familiar.
- `Deck`: primera letra de la cabina; cuando falta cabina se marca como `Unknown`.


In [ ]:
def add_features(data: pd.DataFrame) -> pd.DataFrame:
    """Crea variables ?tiles para el an?lisis y el modelo."""
    data = data.copy()
    data["Title"] = data["Name"].str.extract(r",\s*([^\.]+)\.", expand=False).fillna("Unknown")
    data["FamilySize"] = data["SibSp"] + data["Parch"] + 1
    data["IsAlone"] = (data["FamilySize"] == 1).astype(int)
    data["FarePerPerson"] = data["Fare"] / data["FamilySize"].replace(0, np.nan)
    data["Deck"] = data["Cabin"].str[0].fillna("Unknown")
    return data

df_model = add_features(df)
df_model[["Name", "Title", "FamilySize", "IsAlone", "Fare", "FarePerPerson", "Cabin", "Deck"]].head(10)


In [ ]:
# Nuevas variables frente a supervivencia
feature_view = df_model.groupby("Title", dropna=False).agg(
    pasajeros=("PassengerId", "count"),
    supervivencia_media=("Survived", "mean"),
    edad_media=("Age", "mean"),
    tarifa_media=("Fare", "mean"),
).sort_values("pasajeros", ascending=False)

feature_view.head(12).round(3)


## 8. Preparaci?n para modelado

Separaremos la variable objetivo (`Survived`) de las variables predictoras. Para evitar identificadores o texto con demasiada cardinalidad, excluimos columnas como `PassengerId`, `Name`, `Ticket` y `Cabin`, pero aprovechamos variables derivadas m?s compactas como `Title` y `Deck`.

El pipeline hace tres tareas de forma reproducible:

- Imputa valores faltantes.
- Escala variables num?ricas.
- Codifica variables categ?ricas con One-Hot Encoding.


In [ ]:
target = "Survived"
features = [
    "Pclass",
    "Sex",
    "Age",
    "SibSp",
    "Parch",
    "Fare",
    "Embarked",
    "Title",
    "FamilySize",
    "IsAlone",
    "FarePerPerson",
    "Deck",
]

X = df_model[features]
y = df_model[target]

numeric_features = ["Age", "SibSp", "Parch", "Fare", "FamilySize", "IsAlone", "FarePerPerson"]
categorical_features = ["Pclass", "Sex", "Embarked", "Title", "Deck"]

numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numeric_features),
        ("cat", categorical_pipeline, categorical_features),
    ]
)

model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(max_iter=1000, random_state=42)),
    ]
)


In [ ]:
# Divisi?n entrenamiento/prueba con estratificaci?n para conservar la proporci?n de clases
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y,
)

print(f"Entrenamiento: {X_train.shape[0]} filas")
print(f"Prueba: {X_test.shape[0]} filas")


## 9. Entrenamiento y evaluaci?n del modelo

Entrenamos una regresi?n log?stica porque es interpretable, r?pida y apropiada para una variable objetivo binaria. Adem?s de medir exactitud, revisamos matriz de confusi?n, reporte de clasificaci?n y AUC ROC.


In [ ]:
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

accuracy = accuracy_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_proba)

print(f"Accuracy: {accuracy:.3f}")
print(f"AUC ROC: {auc:.3f}\n")
print(classification_report(y_test, y_pred, target_names=["No sobrevivi?", "Sobrevivi?"]))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred,
    display_labels=["No", "S?"],
    cmap="Blues",
    ax=axes[0],
    colorbar=False,
)
axes[0].set_title("Matriz de confusi?n")

RocCurveDisplay.from_predictions(y_test, y_proba, ax=axes[1])
axes[1].plot([0, 1], [0, 1], linestyle="--", color="gray")
axes[1].set_title("Curva ROC")

plt.tight_layout()
plt.show()


In [ ]:
# Validaci?n cruzada para tener una estimaci?n m?s estable
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(model, X, y, cv=cv, scoring="accuracy")

print(f"Accuracy CV promedio: {cv_scores.mean():.3f}")
print(f"Desviaci?n est?ndar CV: {cv_scores.std():.3f}")
print(f"Scores por fold: {np.round(cv_scores, 3)}")


## 10. Interpretaci?n de variables

La regresi?n log?stica permite revisar qu? variables empujan la predicci?n hacia mayor o menor probabilidad de supervivencia. Los coeficientes deben interpretarse con cautela porque las variables categ?ricas fueron transformadas con One-Hot Encoding y las num?ricas fueron escaladas.


In [ ]:
# Extraer nombres de variables transformadas y coeficientes del modelo
preprocessor_fitted = model.named_steps["preprocessor"]
classifier = model.named_steps["classifier"]

encoded_categorical = preprocessor_fitted.named_transformers_["cat"].named_steps["onehot"].get_feature_names_out(categorical_features)
feature_names = numeric_features + list(encoded_categorical)

coef_table = (
    pd.DataFrame({"variable": feature_names, "coeficiente": classifier.coef_[0]})
    .assign(abs_coef=lambda x: x["coeficiente"].abs())
    .sort_values("abs_coef", ascending=False)
    .drop(columns="abs_coef")
)

coef_table.head(15)


## 11. Predicciones finales

Creamos una tabla compacta con el identificador del pasajero, la predicci?n del modelo, la probabilidad estimada y algunas variables ?tiles para revisar resultados.


In [ ]:
all_predictions = df_model[["PassengerId", "Name", "Sex", "Pclass", "Age", "Fare", "Survived"]].copy()
all_predictions["PredictedSurvived"] = model.predict(X)
all_predictions["SurvivalProbability"] = model.predict_proba(X)[:, 1]

all_predictions.sort_values("SurvivalProbability", ascending=False).head(15)


In [ ]:
# Guardar predicciones para uso posterior
OUTPUT_PATH = Path("titanic_predictions.csv")
all_predictions.to_csv(OUTPUT_PATH, index=False)
print(f"Archivo generado: {OUTPUT_PATH.resolve()}")


## 12. Conclusiones

1. El dataset contiene 418 pasajeros y 12 columnas originales. Las variables con m?s valores faltantes son `Cabin`, `Age` y `Fare`.
2. La supervivencia observada est? fuertemente asociada con el sexo registrado del pasajero. Esto produce un modelo con m?tricas potencialmente muy altas, pero tambi?n exige cautela: puede tratarse de una etiqueta construida o de una regla de supervivencia simplificada dentro del archivo.
3. La ingenier?a de variables permite enriquecer el an?lisis con atributos como tama?o de familia, t?tulo social y cubierta de cabina.
4. La regresi?n log?stica es una buena primera aproximaci?n porque combina rendimiento aceptable e interpretabilidad.
5. Para un proyecto acad?mico o profesional, el siguiente paso recomendado ser?a comparar este modelo con ?rboles de decisi?n, Random Forest o Gradient Boosting, y validar contra una fuente con etiquetas hist?ricas confirmadas.

Este cuaderno queda listo para ejecutarse de principio a fin y producir una tabla final de predicciones en `titanic_predictions.csv`.
